# Práctica 3 - Análisis de Resultados

En este notebook se procesan los resultados obtenidos en la etapa de evaluación (`eval.ipynb`). Se generarán:
1. Tablas resumen por cada método (filas: datasets, columnas: métricas).
2. Tabla resumen específica para la métrica F1-Score.
3. Gráficas comparativas (FN vs FP, PR vs RC, Acc vs Fm).

 Importación y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Directorios
results_dir = 'resultados_eval'
csv_path = os.path.join(results_dir, 'evaluacion_resultados.csv')

# Cargar datos
df = pd.read_csv(csv_path)

# Definir el orden lógico de los datasets para las tablas (no alfabético)
orden_datasets = [
    'original', 'estandarizados', 'normalizados',
    'originalPCA95', 'originalPCA80',
    'estandarizadoPCA95', 'estandarizadoPCA80',
    'normalizadoPCA95', 'normalizadoPCA80'
]

# Definir el orden de los modelos
orden_modelos = [
    'KNN', 'SVM', 'NaiveBayes', 'RandomForest', 
    'Ensemble_Voting', 'Ensemble_Mean', 'Ensemble_Median'
]

# Verificar carga
print(f"Registros cargados: {len(df)}")
df.head()

Agregación de Resultados (Media y Desviación Típica)

In [1]:
# Agrupar por Dataset y Modelo, calculando media y desviación estándar de todas las métricas
metrics_cols = ['Fm', 'Acc', 'S', 'SP', 'RC', 'PR', 'FNR', 'FPR', 'AUC', 'Global_Acc']

# Calculamos la media y la desviación
grouped = df.groupby(['Dataset', 'Model'])[metrics_cols].agg(['mean', 'std'])

# Función para formatear estilo LaTeX "media ± std"
def format_latex(row, metric):
    mean = row[(metric, 'mean')]
    std = row[(metric, 'std')]
    return f"{mean:.4f} $\\pm$ {std:.4f}"

# Crear un DataFrame formateado para visualizar
df_formatted = pd.DataFrame(index=grouped.index)
for col in metrics_cols:
    df_formatted[col] = grouped.apply(lambda x: format_latex(x, col), axis=1)

df_formatted.reset_index(inplace=True)
print("Datos agregados y formateados correctamente.")

NameError: name 'df' is not defined

Tablas Resumen por Método

In [2]:
# Iterar sobre cada modelo para mostrar (y guardar) su tabla
unique_models = df_formatted['Model'].unique()

# Ordenar según nuestra lista preferida si existen
models_to_show = [m for m in orden_modelos if m in unique_models]

for model in models_to_show:
    print(f"\n{'='*20} TABLA RESUMEN: {model} {'='*20}")
    
    # Filtrar por modelo
    subset = df_formatted[df_formatted['Model'] == model].copy()
    
    # Ordenar datasets según el orden lógico
    subset['Dataset'] = pd.Categorical(subset['Dataset'], categories=orden_datasets, ordered=True)
    subset = subset.sort_values('Dataset')
    
    # Seleccionar columnas a mostrar (quitamos 'Model' porque es redundante aquí)
    display_cols = ['Dataset'] + metrics_cols
    display_df = subset[display_cols]
    
    # Mostrar en el notebook
    display(display_df)
    
    # Opcional: Guardar a CSV o imprimir código LaTeX
    # print(display_df.to_latex(index=False, escape=False))

NameError: name 'df_formatted' is not defined

Tabla Resumen F1-Score

In [ ]:
print(f"\n{'='*20} TABLA RESUMEN: F1-SCORE (Fm) {'='*20}")

# Pivotar la tabla: Índices=Modelos, Columnas=Datasets, Valores=Fm formateado
f1_summary = df_formatted.pivot(index='Model', columns='Dataset', values='Fm')

# Reordenar filas y columnas
f1_summary = f1_summary.reindex(index=orden_modelos, columns=orden_datasets)

# Mostrar
display(f1_summary)

# Guardar
f1_summary.to_csv(os.path.join(results_dir, 'summary_f1_score.csv'))

Generación de Gráficas

In [ ]:
# Preparamos los datos numéricos (usamos las medias calculadas en 'grouped')
# grouped tiene MultiIndex en columnas (metric, stat). Nos quedamos solo con 'mean'.
df_means = grouped.xs('mean', axis=1, level=1).reset_index()

# Filtramos solo los modelos principales y ensembles para no saturar, o todos.
# Añadimos una columna 'Type' para diferenciar visualmente Modelos base de Ensembles
def get_type(name):
    if 'Ensemble' in name: return 'Ensemble'
    return 'Single Model'

df_means['Type'] = df_means['Model'].apply(get_type)

# Definimos una paleta de colores consistente
palette = sns.color_palette("husl", len(df_means['Model'].unique()))

# --- FIGURA 1: FN contra FP (Usaremos las tasas FNR vs FPR) ---
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df_means, 
    x='FPR', y='FNR', 
    hue='Model', style='Dataset', 
    s=100, palette=palette, alpha=0.8
)
plt.title('Tasa Falsos Negativos (FNR) vs Tasa Falsos Positivos (FPR)')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('False Negative Rate (FNR)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'plot_FNR_vs_FPR.png'))
plt.show()

# --- FIGURA 2: PR contra RC (Precision vs Recall) ---
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df_means, 
    x='RC', y='PR', 
    hue='Model', style='Dataset', 
    s=100, palette=palette, alpha=0.8
)
plt.title('Precisión (PR) vs Recall (RC)')
plt.xlabel('Recall (Sensibilidad)')
plt.ylabel('Precision')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'plot_PR_vs_RC.png'))
plt.show()

# --- FIGURA 3: ACC contra Fm (Accuracy vs F1-Score) ---
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df_means, 
    x='Fm', y='Acc', 
    hue='Model', style='Dataset', 
    s=100, palette=palette, alpha=0.8
)
plt.title('Exactitud (Accuracy) vs F1-Score')
plt.xlabel('F1-Score')
plt.ylabel('Accuracy (Binaria promedio)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'plot_Acc_vs_Fm.png'))
plt.show()